In [18]:
# Importaciones necesarias
import pandas as pd
from sklearn.model_selection import train_test_split

In [19]:
# Cargar datos
df = pd.read_csv("df_encoded.csv")

# Cargar un conjunto de datos de clasificación
X = df.drop(['puntaje_cat'], axis=1)
y = df["puntaje_cat"]

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

##### Importar MLFlow para registrar los experimentos, el regresor de bosques aleatorios y la métrica de error cuadrático medio

In [20]:
# Importar
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Configurar MLflow
mlflow.set_tracking_uri('http://localhost:5000')
experiment = mlflow.set_experiment("sklearn-clas-icfes")

# Registro del experimento
with mlflow.start_run(experiment_id=experiment.experiment_id):
    # Hiperparámetros
    n_estimators = 80
    max_depth = 10
    max_features = 'log2'
    min_samples_split = 5
    min_samples_leaf = 3
    class_weight = 'balanced'
    
    # Modelo de clasificación
    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_features=max_features,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        class_weight=class_weight,
        random_state=42
    )
    
    # Entrenamiento
    clf.fit(X_train, y_train)
    
    # Predicción
    predictions = clf.predict(X_test)
    
    # Registrar parámetros
    mlflow.log_param("num_trees", n_estimators)
    mlflow.log_param("maxdepth", max_depth)
    mlflow.log_param("max_feat", max_features)
    mlflow.log_param("min_samples_split", min_samples_split)
    mlflow.log_param("min_samples_leaf", min_samples_leaf)
    mlflow.log_param("class_weight", class_weight)
    
    # Registrar modelo
    mlflow.sklearn.log_model(clf, "random-forest-classifier")
    
    # Registrar métricas
    accuracy = accuracy_score(y_test, predictions)
    mlflow.log_metric("accuracy", accuracy)
    
    print(f"Accuracy: {accuracy}")
    print(classification_report(y_test, predictions))


2025/05/25 17:31:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Accuracy: 0.692868358958426
              precision    recall  f1-score   support

           0       0.70      0.70      0.70     61308
           1       0.68      0.69      0.69     57781

    accuracy                           0.69    119089
   macro avg       0.69      0.69      0.69    119089
weighted avg       0.69      0.69      0.69    119089

🏃 View run angry-bird-953 at: http://localhost:5000/#/experiments/951439129855683819/runs/ebe2769fa80c4f23b1b8121cb21362ee
🧪 View experiment at: http://localhost:5000/#/experiments/951439129855683819
